In [ ]:
import os, sys, time
import numpy as np, pandas as pd
from scipy.stats import rankdata

In [1]:
"""
make_full_blend.py  --  reproduces submission_full_blend.csv (public LB 0.95859)
 
Final = 0.70 * rank(BASE) + 0.30 * rank(CV_LGB)
 
  BASE   = 0.86*GBDT + 0.091*TabICL_50k + 0.049*TabICL_100k
  GBDT   = 0.75*(0.57*lgb_optimized + 0.24*baseline_xgb + 0.19*baseline_cat) + 0.25*lgb_specialized_service
  CV_LGB = rank-average (50/50) of two LightGBMs (all features / no delay columns),
           each trained with 5-fold stratified CV; test predictions averaged over the 5 fold models.
 
Required files in DATA_DIR:
  train.csv, test.csv
  lgb_optimized.npy, baseline_xgb.npy, baseline_cat.npy, lgb_specialized_service.npy   (test predictions, 299,844 rows)
  tabicl_50k_test.npy, tabicl_100k_test.npy                                            (test predictions, 299,844 rows)
 
Usage:  python make_full_blend.py [DATA_DIR] [OUT_CSV]
The CV step takes ~10 min on 1 CPU; its result is cached in cv_test_preds.npy and reused on later runs
(delete that file to retrain).   pip install lightgbm scikit-learn scipy pandas numpy
"""

'\nmake_full_blend.py  --  reproduces submission_full_blend.csv (public LB 0.95859)\n\nFinal = 0.70 * rank(BASE) + 0.30 * rank(CV_LGB)\n\n  BASE   = 0.86*GBDT + 0.091*TabICL_50k + 0.049*TabICL_100k\n  GBDT   = 0.75*(0.57*lgb_optimized + 0.24*baseline_xgb + 0.19*baseline_cat) + 0.25*lgb_specialized_service\n  CV_LGB = rank-average (50/50) of two LightGBMs (all features / no delay columns),\n           each trained with 5-fold stratified CV; test predictions averaged over the 5 fold models.\n\nRequired files in DATA_DIR:\n  train.csv, test.csv\n  lgb_optimized.npy, baseline_xgb.npy, baseline_cat.npy, lgb_specialized_service.npy   (test predictions, 299,844 rows)\n  tabicl_50k_test.npy, tabicl_100k_test.npy                                            (test predictions, 299,844 rows)\n\nUsage:  python make_full_blend.py [DATA_DIR] [OUT_CSV]\nThe CV step takes ~10 min on 1 CPU; its result is cached in cv_test_preds.npy and reused on later runs\n(delete that file to retrain).   pip install li

In [ ]:
DATA_DIR = sys.argv[1] if len(sys.argv) > 1 else "/mnt/user-data/uploads"
OUT_CSV  = sys.argv[2] if len(sys.argv) > 2 else "submission_full_blend.csv"
CACHE    = "cv_test_preds.npy"
W_CV     = 0.30                                   # weight of the CV-LightGBM component
rank = lambda a: rankdata(a) / len(a)
P = lambda f: os.path.join(DATA_DIR, f)

In [ ]:
def run_cv(train, test):
    import lightgbm as lgb
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score

    CAT   = ["Gender", "Customer Type", "Type of Travel", "Class"]
    DELAY = ["Departure Delay in Minutes", "Arrival Delay in Minutes"]
    feats = [c for c in test.columns if c != "id"]
    y = train["satisfaction"].astype(int).values

    def prep(df, cols):
        d = df[cols].copy()
        for c in CAT:
            if c in d:
                d[c] = pd.Categorical(d[c], categories=sorted(train[c].unique()))
        return d

    variants = {
        "lgb_full":    dict(cols=feats, params=dict(num_leaves=127, colsample_bytree=0.85, random_state=42)),
        "lgb_nodelay": dict(cols=[c for c in feats if c not in DELAY],
                            params=dict(num_leaves=127, colsample_bytree=0.80, random_state=7)),
    }
    base = dict(objective="binary", n_estimators=4000, learning_rate=0.03, min_child_samples=20,
                reg_alpha=0.1, reg_lambda=0.5, n_jobs=-1, verbosity=-1)

    skf = StratifiedKFold(5, shuffle=True, random_state=2026)
    oof = {k: np.zeros(len(train)) for k in variants}
    tp  = {k: np.zeros(len(test))  for k in variants}
    for fold, (tr, va) in enumerate(skf.split(train, y)):
        for k, v in variants.items():
            t = time.time()
            Xa, Xv, Xt = prep(train.iloc[tr], v["cols"]), prep(train.iloc[va], v["cols"]), prep(test, v["cols"])
            m = lgb.LGBMClassifier(**base, **v["params"])
            m.fit(Xa, y[tr], eval_set=[(Xv, y[va])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(100, verbose=False)])
            oof[k][va] = m.predict_proba(Xv)[:, 1]
            tp[k] += m.predict_proba(Xt)[:, 1] / 5
            print(f"fold {fold} {k}: AUC {roc_auc_score(y[va], oof[k][va]):.6f}  iters {m.best_iteration_}  {time.time()-t:.0f}s", flush=True)
    print("OOF AUC:", {k: round(roc_auc_score(y, oof[k]), 6) for k in oof})
    return rank(0.5 * rank(tp["lgb_full"]) + 0.5 * rank(tp["lgb_nodelay"]))

In [ ]:
def main():
    test = pd.read_csv(P("test.csv"))

    if os.path.exists(CACHE):
        cv_lgb = np.load(CACHE); print("loaded cached CV predictions:", CACHE)
    else:
        train = pd.read_csv(P("train.csv"))
        cv_lgb = run_cv(train, test); np.save(CACHE, cv_lgb)

    lgb_o = np.load(P("lgb_optimized.npy"))
    xgb   = np.load(P("baseline_xgb.npy")).astype(float)
    cat   = np.load(P("baseline_cat.npy"))
    spec  = np.load(P("lgb_specialized_service.npy"))
    t50   = np.load(P("tabicl_50k_test.npy"))
    t100  = np.load(P("tabicl_100k_test.npy"))
    for n, a in dict(lgb_o=lgb_o, xgb=xgb, cat=cat, spec=spec, t50=t50, t100=t100, cv=cv_lgb).items():
        assert len(a) == len(test), f"{n}: {len(a)} rows, expected {len(test)}"

    gbdt = 0.75 * (0.57 * lgb_o + 0.24 * xgb + 0.19 * cat) + 0.25 * spec
    base = 0.86 * gbdt + 0.091 * t50 + 0.049 * t100
    final = (1 - W_CV) * rank(base) + W_CV * cv_lgb

    sub = pd.DataFrame({"id": test["id"], "satisfaction": final})
    assert sub["satisfaction"].notna().all()
    sub.to_csv(OUT_CSV, index=False)
    print("saved", OUT_CSV, sub.shape)

In [ ]:
if __name__ == "__main__":
    main()